# 99_run — Black-Litterman 실험 실행

**역할**: bl_config.py에 정의된 모든 실험을 walk-forward로 실행하고 결과를 `results/` 폴더에 저장.

## 실행 순서
1. 데이터 로드
2. Dispatcher 함수 정의 (슬롯별 config → 함수 분기)
3. walk_forward() 함수 정의
4. 전체 실험 실행 → pkl 저장

> 분석/시각화는 **99_analyze.ipynb**에서.

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import platform
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

from bl_functions import (
    compute_sigma, compute_daily_slice, compute_pi,
    build_P,
    compute_Q_fixed, compute_Q_ff3, compute_Q_spread, compute_Q_regime,
    compute_omega_he, compute_omega_scaled, compute_omega_rmse,
    black_litterman, optimize_portfolio,
    compute_turnover, apply_tc, compute_metrics,
)
from bl_config import EXPERIMENTS, get_changed_slots

BASE_DIR    = Path.cwd()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), f'data/ 폴더 없음. 01_DataCollection.ipynb 먼저 실행하세요.\n경로: {DATA_DIR}'

# ── 공통 파라미터 ──────────────────────────────────────────────
TRAIN_WINDOW  = 60
THRESH_DAILY  = 0.9
TAU           = 0.1
PCT_GROUP     = 0.30
START_PRED    = '2010-01-01'

print(f'설정 완료')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  실험 수     : {len(EXPERIMENTS)}개')

설정 완료
  DATA_DIR    : c:\workspace\camp\project\finance_project\final\data
  RESULTS_DIR : c:\workspace\camp\project\finance_project\final\results
  실험 수     : 9개


In [2]:
# ── 데이터 로드 ───────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()
ret_pivot  = panel['ret_1m'].unstack('ticker')   # FF3 Q 계산용

# FF3 팩터
import io, re, zipfile, requests
FF3_PATH = DATA_DIR / 'ff3_monthly.csv'
if FF3_PATH.exists():
    ff3 = pd.read_csv(FF3_PATH, index_col='date', parse_dates=True)
else:
    print('FF3 다운로드 중...')
    url  = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip'
    resp = requests.get(url, timeout=60)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0]).decode('utf-8', errors='ignore')
    lines = raw.splitlines()
    start = next(i for i, l in enumerate(lines) if re.match(r'^\s*\d{6}\s*,', l))
    end   = next((i for i in range(start, len(lines)) if not re.match(r'^\s*\d{6}\s*,', lines[i])), len(lines))
    df    = pd.read_csv(io.StringIO('\n'.join(lines[start-1:end])))
    df.columns = [c.strip() for c in df.columns]
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
    ff3 = df.rename(columns={dc:'date','Mkt-RF':'mkt_rf','SMB':'smb','HML':'hml','RF':'rf'}).set_index('date').astype(float) / 100.0
    ff3.to_csv(FF3_PATH)

print(f'패널       : {panel.shape}')
print(f'일별 수익률: {daily_ret.shape}')
print(f'예측 기간  : {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

패널       : (97944, 11)
일별 수익률: (5595, 822)
예측 기간  : 2010-01-31 ~ 2024-12-31 (180개월)


In [3]:
# ── LSTM 예측값 로드 (없으면 None) ────────────────────────────
# bl_config.py의 BASELINE['lstm_pred_path']에 경로 지정
# 파일이 없으면 LSTM 관련 실험은 자동 스킵

from bl_config import BASELINE

_lstm_path = Path(BASELINE.get('lstm_pred_path', ''))
if _lstm_path.exists():
    lstm_preds = pd.read_csv(_lstm_path, parse_dates=['date'])
    lstm_preds = lstm_preds.set_index(['date', 'ticker'])
    # exp() 변환: log(daily_std) → daily_std
    lstm_preds['vol_pred'] = np.exp(lstm_preds['y_pred_ensemble'])
    # RMSE 계산 (후속 omega_rmse 용)
    lstm_preds['abs_err'] = (lstm_preds['y_pred_ensemble'] - lstm_preds['y_true']).abs()
    print(f'LSTM 예측 로드 완료: {lstm_preds.shape}')
    LSTM_AVAILABLE = True
else:
    lstm_preds = None
    LSTM_AVAILABLE = False
    print(f'⚠  LSTM 예측 파일 없음 → lstm 관련 실험 스킵')
    print(f'   경로: {_lstm_path}')

⚠  LSTM 예측 파일 없음 → lstm 관련 실험 스킵
   경로: c:\workspace\camp\project\finance_project\시계열_test\Phase3_Robust_Extensions\results\ensemble_predictions_stockwise.csv


In [4]:
# ══════════════════════════════════════════════════════════════
# Dispatcher 함수 (config → 실제 함수 호출)
# 새 방식 추가 시 해당 elif만 추가
# ══════════════════════════════════════════════════════════════

def get_vol_series(cfg, month_df, pred_date):
    """P 행렬에 쓸 변동성 시리즈 반환."""
    mode = cfg.get('p_mode', 'trailing_vol21')

    if mode == 'trailing_vol21':
        return month_df['vol_21d']

    elif mode == 'trailing_vol252':
        return month_df['vol_252d']

    elif mode == 'lstm_predicted':
        if lstm_preds is None:
            return month_df['vol_21d']
        if pred_date in lstm_preds.index.get_level_values('date'):
            pred_slice = lstm_preds.xs(pred_date, level='date')['vol_pred']
            vol = month_df['vol_21d'].copy()
            common = vol.index.intersection(pred_slice.index)
            vol[common] = pred_slice[common]
            return vol
        return month_df['vol_21d']

    else:
        raise ValueError(f'p_mode={mode!r} 미지원')


def get_prior_weights(cfg, valid_tix, mcap):
    """prior 시장가중치 w_mkt 반환."""
    prior = cfg.get('prior', 'capm_mcap')
    if prior == 'capm_mcap':
        return (mcap / mcap.sum()).reindex(valid_tix).fillna(0)
    elif prior == 'capm_eq':
        n = len(valid_tix)
        return pd.Series(1.0 / n, index=valid_tix)
    else:
        raise ValueError(f'prior={prior!r} 미지원')


def get_Q(cfg, P, valid_tix, train_dates, pred_date, all_d):
    """Q 값 반환. q_mode='capm'/'none'은 walk_forward에서 직접 처리."""
    mode = cfg.get('q_mode', 'fixed')

    if mode == 'fixed':
        return compute_Q_fixed(cfg.get('q_value', 0.003))

    elif mode == 'ff3_regression':
        thresh_m    = int(len(train_dates) * 0.7)
        monthly_sl  = ret_pivot.reindex(index=train_dates, columns=valid_tix).dropna(axis=1, thresh=thresh_m)
        ff3_train   = ff3.reindex(train_dates)
        rf_train    = rf_series.reindex(train_dates)
        return compute_Q_ff3(P.reindex(monthly_sl.columns).fillna(0), monthly_sl, ff3_train, rf_train)

    elif mode == 'realized_spread':
        panel_train = panel[
            (panel.index.get_level_values('date') >= train_dates[0]) &
            (panel.index.get_level_values('date') <= train_dates[-1])
        ]
        return compute_Q_spread(panel_train, pct=PCT_GROUP)

    elif mode == 'regime':
        spy_train = spy_series.reindex(train_dates)
        return compute_Q_regime(spy_train, cfg.get('q_regime_table', {}))

    else:
        raise ValueError(f'q_mode={mode!r} 미지원')


def get_omega(cfg, P, Sigma, pred_date):
    """Omega 값 반환."""
    mode = cfg.get('omega_mode', 'he_litterman')

    if mode == 'he_litterman':
        return compute_omega_he(P, Sigma, TAU)

    elif mode == 'scaled':
        return compute_omega_scaled(P, Sigma, TAU, cfg.get('omega_scale', 1.0))

    elif mode == 'rmse':
        if lstm_preds is None:
            return compute_omega_he(P, Sigma, TAU)
        cutoff = pred_date - pd.DateOffset(months=12)
        recent = lstm_preds[
            (lstm_preds.index.get_level_values('date') >= cutoff) &
            (lstm_preds.index.get_level_values('date') <= pred_date)
        ]
        pred_rmse = float(recent['abs_err'].mean()) if len(recent) > 0 else 0.39
        return compute_omega_rmse(P, Sigma, TAU, pred_rmse)

    else:
        raise ValueError(f'omega_mode={mode!r} 미지원')


print('Dispatcher 함수 정의 완료')

Dispatcher 함수 정의 완료


In [5]:
import time

# ══════════════════════════════════════════════════════════════
# 월별 공유 데이터 캐시 (모든 실험에서 동일한 부분 — 한 번만 계산)
#   Sigma (LedoitWolf), month_df, mcap, train_dates, spy_excess, sigma2_mkt
#   실험별로 달라지는 것: vol_series(p_mode), P(p_weight), Q, omega, w
# ══════════════════════════════════════════════════════════════

monthly_cache = {}
_cache_t0 = time.time()

for i, pred_date in enumerate(pred_dates):
    if i % 24 == 0:
        elapsed = time.time() - _cache_t0
        pct     = (i + 1) / len(pred_dates)
        eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
        print(f'  캐시 {pred_date.strftime("%Y-%m")} ({i+1}/{len(pred_dates)}, {pct:.0%}) | '
              f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)
    try:
        month_df = panel.xs(pred_date, level='date').dropna(
            subset=['vol_21d', 'log_mcap', 'ret_1m'])
        if len(month_df) < 30:
            continue

        daily_slice, valid_tix = compute_daily_slice(
            pred_date, month_df.index.tolist(),
            daily_ret, TRAIN_WINDOW, THRESH_DAILY)
        if len(valid_tix) < 20:
            continue

        Sigma    = compute_sigma(daily_slice, scale=21)
        month_df = month_df.reindex(valid_tix)
        mcap     = np.exp(month_df['log_mcap'])

        idx         = all_dates.get_loc(pred_date)
        train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
        spy_s       = spy_series.reindex(train_dates)
        rf_s        = rf_series.reindex(train_dates)

        next_date = all_dates[idx + 1] if idx + 1 < len(all_dates) else None

        monthly_cache[pred_date] = {
            'month_df'   : month_df,
            'valid_tix'  : valid_tix,
            'Sigma'      : Sigma,
            'mcap'       : mcap,
            'train_dates': train_dates,
            'spy_excess' : float((spy_s - rf_s).mean()),
            'sigma2_mkt' : float(spy_s.var()),
            'next_date'  : next_date,
        }
    except Exception as e:
        print(f'  [캐시 에러] {pred_date.date()}: {e}')

_cache_min = (time.time() - _cache_t0) / 60
print(f'\n캐시 완료: {len(monthly_cache)}개월 / {len(pred_dates)}개월 | {_cache_min:.1f}분 소요')
print('이후 실험은 Sigma 재계산 없이 캐시에서 로드')

  캐시 2010-01 (1/180, 1%) | 경과 0.0분 | ETA 0.0분
  캐시 2012-01 (25/180, 14%) | 경과 0.1분 | ETA 0.7분
  캐시 2014-01 (49/180, 27%) | 경과 0.2분 | ETA 0.5분
  캐시 2016-01 (73/180, 41%) | 경과 0.3분 | ETA 0.5분
  캐시 2018-01 (97/180, 54%) | 경과 0.4분 | ETA 0.4분
  캐시 2020-01 (121/180, 67%) | 경과 0.6분 | ETA 0.3분
  캐시 2022-01 (145/180, 81%) | 경과 0.7분 | ETA 0.2분
  캐시 2024-01 (169/180, 94%) | 경과 0.9분 | ETA 0.1분

캐시 완료: 180개월 / 180개월 | 1.0분 소요
이후 실험은 Sigma 재계산 없이 캐시에서 로드


In [6]:
import time

# ══════════════════════════════════════════════════════════════
# walk_forward(cfg) — 실험 하나 실행 (캐시에서 Sigma 로드)
# ══════════════════════════════════════════════════════════════

def walk_forward(cfg: dict) -> dict:
    name       = cfg['name']
    tc         = cfg.get('tc', 0.001)
    max_w      = cfg.get('max_weight', 0.10)
    q_mode     = cfg.get('q_mode', 'fixed')
    p_weight   = cfg.get('p_weight', 'mcap')
    is_naive   = (q_mode == 'none')
    is_capm    = (q_mode == 'capm')

    ret_list, comp_list, meta_list = [], [], []
    spy_list, err_list = [], []
    weights_history = {}
    prev_w = None
    _t0 = time.time()

    for i, pred_date in enumerate(pred_dates):

        # ── 진행 현황 (12개월마다 + 마지막) ──────────────────
        if i % 12 == 0 or i == len(pred_dates) - 1:
            elapsed = time.time() - _t0
            pct     = (i + 1) / len(pred_dates)
            eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
            print(f'  [{name}] {pred_date.strftime("%Y-%m")} '
                  f'({i+1}/{len(pred_dates)}, {pct:.0%}) | '
                  f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)

        # ── 캐시에서 공유 데이터 로드 ────────────────────────
        if pred_date not in monthly_cache:
            continue
        c           = monthly_cache[pred_date]
        month_df    = c['month_df']
        valid_tix   = c['valid_tix']
        Sigma       = c['Sigma']
        mcap        = c['mcap']
        train_dates = c['train_dates']
        spy_excess  = c['spy_excess']
        sigma2_mkt  = c['sigma2_mkt']
        next_date   = c['next_date']

        try:
            w_mkt   = get_prior_weights(cfg, valid_tix, mcap)
            pi, lam = compute_pi(Sigma, w_mkt, spy_excess, sigma2_mkt)

            vol_series = get_vol_series(cfg, month_df, pred_date)
            P = build_P(
                vol_series.reindex(valid_tix).fillna(vol_series.median()),
                mcap, pct=PCT_GROUP, weighting=p_weight)

            # ── 가중치 계산 ──────────────────────────────────────
            if is_capm:
                w = optimize_portfolio(pi, Sigma, lam, max_w)
                mu_meta = None

            elif is_naive:
                n_g     = max(1, int(len(valid_tix) * PCT_GROUP))
                vol_v   = vol_series.reindex(valid_tix)
                low_tix = vol_v.sort_values().index[:n_g]

                if p_weight == 'mcap':
                    w_low = mcap.reindex(low_tix)
                    w_low = w_low / w_low.sum()
                elif p_weight == 'rp':
                    inv_v = (1.0 / vol_v[low_tix]).replace(0, np.nan).dropna()
                    w_low = inv_v / inv_v.sum()
                else:
                    w_low = pd.Series(1.0 / len(low_tix), index=low_tix)

                w = w_low.reindex(valid_tix).fillna(0)
                w = w.clip(upper=max_w)
                w = w / w.sum()
                mu_meta = None

            else:
                Q     = get_Q(cfg, P, valid_tix, train_dates, pred_date, all_dates)
                omega = get_omega(cfg, P, Sigma, pred_date)
                mu_BL = black_litterman(pi, Sigma, P, Q, omega, TAU)
                w     = optimize_portfolio(mu_BL, Sigma, lam, max_w)
                mu_meta = Q

            actual_ret = month_df['fwd_ret_1m'].reindex(valid_tix).fillna(0)
            gross_ret  = float(w @ actual_ret)
            turnover   = compute_turnover(w, prev_w) if prev_w is not None else 0.0
            net_ret    = apply_tc(gross_ret, turnover, tc)
            r_spy      = float(spy_series.get(next_date, np.nan)) if next_date else np.nan

            ret_list.append({'date': pred_date, 'ret': net_ret, 'gross_ret': gross_ret})
            spy_list.append({'date': pred_date, 'ret': r_spy})

            vol_col      = month_df['vol_21d'].reindex(valid_tix)
            n_g          = max(1, int(len(valid_tix) * PCT_GROUP))
            sv           = vol_col.sort_values()
            low_tix_all  = sv.index[:n_g].tolist()
            high_tix_all = sv.index[-n_g:].tolist()

            comp_list.append({
                'date'        : pred_date,
                'n_stocks'    : len(valid_tix),
                'eff_n'       : 1.0 / float((w**2).sum()) if (w**2).sum() > 0 else 0,
                'top10_share' : float(w.nlargest(10).sum()),
                'top1_weight' : float(w.max()),
                'top1_ticker' : w.idxmax(),
                'avg_vol'     : float((w * vol_col).sum()),
                'low_weight'  : float(w.reindex(low_tix_all).sum()),
                'high_weight' : float(w.reindex(high_tix_all).sum()),
                'turnover'    : turnover,
                'tc_cost'     : turnover * tc,
            })

            meta_list.append({'date': pred_date, 'Q': mu_meta, 'lam': lam})
            weights_history[pred_date] = w
            prev_w = w

        except Exception as e:
            err_list.append({'date': pred_date, 'error': str(e)})
            if len(err_list) <= 3:
                print(f'  [{name}] 에러 {pred_date.date()}: {e}')

    total_min = (time.time() - _t0) / 60
    ret_df  = pd.DataFrame(ret_list).set_index('date')  if ret_list  else pd.DataFrame()
    spy_df  = pd.DataFrame(spy_list).set_index('date')  if spy_list  else pd.DataFrame()
    comp_df = pd.DataFrame(comp_list).set_index('date') if comp_list else pd.DataFrame()
    meta_df = pd.DataFrame(meta_list).set_index('date') if meta_list else pd.DataFrame()
    wts_df  = pd.DataFrame(weights_history).T            if weights_history else pd.DataFrame()

    print(f'  → [{name}] 완료: {len(ret_df)}개월 성공 / {len(err_list)}개 에러 / {total_min:.1f}분 소요')

    return {
        'config'   : cfg,
        'ret'      : ret_df['ret']       if 'ret'       in ret_df.columns else pd.Series(dtype=float),
        'gross_ret': ret_df['gross_ret'] if 'gross_ret' in ret_df.columns else pd.Series(dtype=float),
        'spy_ret'  : spy_df['ret']       if 'ret'       in spy_df.columns else pd.Series(dtype=float),
        'weights'  : wts_df,
        'comp'     : comp_df,
        'meta'     : meta_df,
        'errors'   : err_list,
    }


print('walk_forward() 함수 정의 완료 (캐시 사용)')

walk_forward() 함수 정의 완료 (캐시 사용)


In [7]:
# ══════════════════════════════════════════════════════════════
# 전체 실험 실행 + 저장
# ══════════════════════════════════════════════════════════════

SKIP_IF_EXISTS = True   # True → 이미 저장된 실험은 재실행 않고 스킵

completed, skipped = [], []
_all_t0 = time.time()
run_list = [cfg for cfg in EXPERIMENTS
            if not (SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists())
            and not (cfg.get('p_mode') == 'lstm_predicted' and not LSTM_AVAILABLE)
            and not (cfg.get('omega_mode') == 'rmse' and not LSTM_AVAILABLE)]

print(f'실행 예정: {len(run_list)}개 / 전체 {len(EXPERIMENTS)}개')
print(f'스킵 (기존 결과): {len(EXPERIMENTS) - len(run_list)}개\n')

for j, cfg in enumerate(EXPERIMENTS):
    name     = cfg['name']
    out_path = RESULTS_DIR / f'{name}.pkl'

    if cfg.get('p_mode') == 'lstm_predicted' and not LSTM_AVAILABLE:
        print(f'[SKIP] {name} — LSTM 예측 파일 없음')
        skipped.append(name)
        continue
    if cfg.get('omega_mode') == 'rmse' and not LSTM_AVAILABLE:
        print(f'[SKIP] {name} — LSTM 예측 파일 없음')
        skipped.append(name)
        continue
    if SKIP_IF_EXISTS and out_path.exists():
        print(f'[SKIP] {name} — 이미 저장됨')
        skipped.append(name)
        continue

    done_so_far = len(completed)
    remaining   = len(run_list) - done_so_far
    elapsed_all = time.time() - _all_t0
    avg_per_exp = elapsed_all / done_so_far if done_so_far > 0 else 0
    eta_all     = avg_per_exp * remaining

    print(f'\n[{done_so_far+1}/{len(run_list)}] {name}  |  '
          f'전체 경과 {elapsed_all/60:.1f}분  |  ETA {eta_all/60:.1f}분')

    result = walk_forward(cfg)

    with open(out_path, 'wb') as f:
        pickle.dump(result, f)

    completed.append(name)

total_elapsed = (time.time() - _all_t0) / 60
print(f'\n{"="*60}')
print(f'완료: {len(completed)}개 / 스킵: {len(skipped)}개 / 총 {total_elapsed:.1f}분')
print(f'저장 위치: {RESULTS_DIR}')

실행 예정: 7개 / 전체 9개
스킵 (기존 결과): 2개

[SKIP] baseline — 이미 저장됨
[SKIP] prior_eq — 이미 저장됨

[1/7] p_vol252  |  전체 경과 0.0분  |  ETA 0.0분
  [p_vol252] 2010-01 (1/180, 1%) | 경과 0.0분 | ETA 0.0분
  [p_vol252] 2011-01 (13/180, 7%) | 경과 1.1분 | ETA 14.2분
  [p_vol252] 2012-01 (25/180, 14%) | 경과 1.8분 | ETA 11.3분
  [p_vol252] 2013-01 (37/180, 21%) | 경과 2.4분 | ETA 9.4분
  [p_vol252] 2014-01 (49/180, 27%) | 경과 3.2분 | ETA 8.4분
  [p_vol252] 2015-01 (61/180, 34%) | 경과 3.8분 | ETA 7.4분
  [p_vol252] 2016-01 (73/180, 41%) | 경과 4.8분 | ETA 7.0분
  [p_vol252] 2017-01 (85/180, 47%) | 경과 5.4분 | ETA 6.0분
  [p_vol252] 2018-01 (97/180, 54%) | 경과 6.0분 | ETA 5.1분
  [p_vol252] 2019-01 (109/180, 61%) | 경과 6.8분 | ETA 4.4분
  [p_vol252] 2020-01 (121/180, 67%) | 경과 7.4분 | ETA 3.6분
  [p_vol252] 2021-01 (133/180, 74%) | 경과 8.2분 | ETA 2.9분
  [p_vol252] 2022-01 (145/180, 81%) | 경과 8.9분 | ETA 2.2분
  [p_vol252] 2023-01 (157/180, 87%) | 경과 9.6분 | ETA 1.4분
  [p_vol252] 2024-01 (169/180, 94%) | 경과 10.2분 | ETA 0.7분
  [p_vol252] 2024-12 (180/

In [8]:
# ── 빠른 결과 확인 ────────────────────────────────────────────
rf_monthly  = rf_series.copy()
spy_monthly = spy_series.copy()

print('=' * 90)
print(f"{'실험명':<25} {'Sharpe':>7} {'Sortino':>8} {'CAGR':>7} {'변동성':>7} {'MDD':>8} {'Beta':>6} {'Alpha':>7}")
print('-' * 90)

for pkl_file in sorted(RESULTS_DIR.glob('*.pkl')):
    with open(pkl_file, 'rb') as f:
        res = pickle.load(f)
    ret = res['ret']
    if len(ret) == 0:
        continue
    m = compute_metrics(ret, rf_monthly, res['config']['name'], mkt_ret=spy_monthly)
    beta_str  = f"{m['beta']:>6.3f}"  if not (isinstance(m['beta'],  float) and np.isnan(m['beta']))  else '   N/A'
    alpha_str = f"{m['alpha']:>7.2%}" if not (isinstance(m['alpha'], float) and np.isnan(m['alpha'])) else '    N/A'
    print(f"{m['label']:<25} {m['sharpe']:>7.3f} {m['sortino']:>8.3f} "
          f"{m['cagr']:>7.2%} {m['vol']:>7.2%} {m['mdd']:>8.2%} "
          f"{beta_str} {alpha_str}")

print('=' * 90)

실험명                        Sharpe  Sortino    CAGR     변동성      MDD   Beta   Alpha
------------------------------------------------------------------------------------------
baseline                    1.106    1.726  13.37%  10.98%  -13.03% -0.136  13.95%
capm_no_bl                  0.899    1.268  14.78%  15.13%  -22.19% -0.149  15.52%
naive_lowvol                1.061    1.549  13.86%  11.91%  -14.73% -0.125  14.30%
naive_lowvol_rp             1.017    1.450  13.47%  12.04%  -14.84% -0.131  13.98%
p_eq                        1.028    1.581  12.38%  10.86%  -12.85% -0.124  12.80%
p_rp                        0.924    1.528  11.79%  11.45%  -13.96% -0.111  12.05%
p_vol252                    1.073    1.701  13.11%  11.09%  -13.10% -0.141  13.75%
p_vol_mcap                  1.013    1.707  11.99%  10.62%  -14.46% -0.113  12.27%
prior_eq                    1.105    1.658  14.22%  11.75%  -13.86% -0.133  14.76%
